# SMVT Phenobarbital 100ns MD — GPU Pilot
**Target**: SMVT (SLC5A6) | **Method**: AMBER ff14SB + GAFF2 + TIP3P  
**Compound**: Phenobarbital (barbiturate hit #1, docking score -8.30 kcal/mol)  
**Simulation**: 100ns NPT, 310K, 1 atm  
---
### Setup (run once)

In [ ]:
!pip install -q openmm openmmforcefields pdbfixer rdkit MDAnalysis matplotlib numpy
import openmm as mm
print('OpenMM', mm.__version__)
print('Platform:', mm.Platform.getPlatformByName('CUDA').getName())

### Upload Input Files
Use the Colab file browser (left sidebar folder icon) to upload to `/content/`:
- `AF-Q9Y289-F1_prepared.pdb` (from `02_Data/cleaned/` — OpenMM minimized receptor)
- `PHENOBARBITAL_docked.pdbqt` (from `03_Analysis/docking/` — Vina docked pose)

If uploading is slow, run the cell below and paste the files into the upload dialog.

In [ ]:
import os
from google.colab import files

DATA_DIR = '/content'
os.makedirs(DATA_DIR, exist_ok=True)

# Upload dialog
uploaded = files.upload()
for filename in uploaded.keys():
    print(f'Uploaded: {filename} ({len(uploaded[filename])} bytes)')

# Verify
required = ['AF-Q9Y289-F1_prepared.pdb', 'PHENOBARBITAL_docked.pdbqt']
for f in required:
    path = os.path.join(DATA_DIR, f)
    if os.path.exists(path):
        print(f'  OK  {f}')
    else:
        print(f'  MISSING  {f} — upload via the dialog above')

---
## MD Pipeline — Phenobarbital 100ns

In [ ]:
import numpy as np
import openmm as mm
import openmm.app as app
import openmm.unit as unit
from openmmforcefields.generators import SystemGenerator
from pdbfixer import PDBFixer
from rdkit import Chem
from rdkit.Chem import AllChem
import time, json, os

# ═══ Config ═══
DATA_DIR = '/content'
OUT_DIR = '/content/trajectories'
os.makedirs(OUT_DIR, exist_ok=True)

RECEPTOR_PDB = f'{DATA_DIR}/AF-Q9Y289-F1_prepared.pdb'

COMPOUND = ('PHENOBARBITAL', 'CCC1(C(=O)NC(=O)NC1=O)C2=CC=CC=C2')

PROD_NS = 100  # 100ns production
TEMP = 310 * unit.kelvin  # physiological temperature
PRESSURE = 1.0 * unit.atmosphere
DT = 2.0 * unit.femtoseconds
CUTOFF = 1.0 * unit.nanometer
SAVE_PS = 100  # save every 100ps

print(f'GPU MD: {COMPOUND[0]} x {PROD_NS}ns (GAFF2, TIP3P, 310K)')

In [ ]:
def extract_vina_pose(pdbqt_path):
    """Extract first-model coordinates from Vina PDBQT output."""
    pos = []; in_model = False
    with open(pdbqt_path) as f:
        for line in f:
            if line.startswith('MODEL'):
                if int(line.split()[1]) == 1:
                    in_model = True; pos = []
                elif in_model:
                    break
            elif line.startswith('ENDMDL') and in_model:
                break
            elif in_model and (line.startswith('ATOM') or line.startswith('HETATM')):
                try:
                    pos.append([float(line[30:38]), float(line[38:46]), float(line[46:54])])
                except:
                    continue
    return pos

def prepare_ligand_pdb(smiles, vina_pos, out_path):
    """Generate ligand PDB from SMILES + Vina docking pose using GAFF2 template."""
    if os.path.exists(out_path):
        return out_path
    print('  Building ligand from SMILES + Vina pose...')
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol, randomSeed=42)
    AllChem.MMFFOptimizeMolecule(mol)
    conf = mol.GetConformer()
    for i in range(min(mol.GetNumAtoms(), len(vina_pos))):
        conf.SetAtomPosition(i, (vina_pos[i][0]/10.0, vina_pos[i][1]/10.0, vina_pos[i][2]/10.0))
    Chem.MolToPDBFile(mol, out_path)
    return out_path

In [ ]:
name, smiles = COMPOUND
tag = name
out_dir = f'{OUT_DIR}/{tag}'
os.makedirs(out_dir, exist_ok=True)
chk_file = f'{out_dir}/{tag}_{PROD_NS}ns.chk'

sep = '=' * 50
print(f'{sep}')
print(f'[{name}] {PROD_NS}ns GPU MD (GAFF2, TIP3P, 310K)')
print(f'{sep}')
t_start = time.time()

# 1. Protein is pre-minimized (OpenMM), skip PDBFixer heavy work
# Just verify and add missing hydrogens if needed
prot_clean = f'{out_dir}/protein.pdb'
if not os.path.exists(prot_clean):
    print('  Preparing protein...')
    fixer = PDBFixer(filename=RECEPTOR_PDB)
    fixer.findMissingResidues()
    fixer.findMissingAtoms()
    fixer.addMissingAtoms()
    fixer.addMissingHydrogens(pH=7.4)
    with open(prot_clean, 'w') as f:
        app.PDBFile.writeFile(fixer.topology, fixer.positions, f)
else:
    print('  Protein already prepared.')

# 2. Extract Vina docking pose → ligand PDB
pdbqt = f'{DATA_DIR}/{name}_docked.pdbqt'
assert os.path.exists(pdbqt), f'PDBQT file missing: {pdbqt}'
vina_pos = extract_vina_pose(pdbqt)
print(f'  Vina pose: {len(vina_pos)} atoms')
lig_pdb = f'{out_dir}/ligand.pdb'
prepare_ligand_pdb(smiles, vina_pos, lig_pdb)

# 3. SystemGenerator with GAFF2
print('  Parameterizing with GAFF2...')
system_gen = SystemGenerator(
    forcefields=['amber/ff14SB.xml', 'amber/tip3p_standard.xml'],
    small_molecule_forcefield='gaff-2.11',
    cache=f'{out_dir}/cache.json'
)

# 4. Build solvated system (TIP3P, 0.15M NaCl, 12A buffer)
print('  Building solvated system...')
prot_pdb = app.PDBFile(prot_clean)
lig = app.PDBFile(lig_pdb)
modeller = app.Modeller(prot_pdb.topology, prot_pdb.positions)
modeller.add(lig.topology, lig.positions)
modeller.addSolvent(system_gen.forcefield, model='tip3p',
                    padding=0.8*unit.nanometers, ionicStrength=0.15*unit.molar,
                    neutralize=True)
system = system_gen.create_system(modeller.topology)
print(f'  System: {system.getNumParticles()} particles')

# 5. Simulation setup (CUDA, mixed precision)
integrator = mm.LangevinMiddleIntegrator(TEMP, 1.0/unit.picosecond, DT)
platform = mm.Platform.getPlatformByName('CUDA')
simulation = app.Simulation(modeller.topology, system, integrator, platform,
                            {'Precision': 'mixed'})
simulation.context.setPositions(modeller.positions)

# 6. Energy minimization
print('  Minimizing...')
simulation.minimizeEnergy(maxIterations=5000)
energy = simulation.context.getState(getEnergy=True).getPotentialEnergy()
print(f'  Minimized energy: {energy}')

# 7. NVT heating (50→150→310K, 30ps total, 1fs timestep)
print('  NVT heating (50→150→310K)...')
simulation.integrator.setStepSize(1.0*unit.femtoseconds)
for t in [50, 150, 310]:
    simulation.integrator.setTemperature(t*unit.kelvin)
    simulation.step(int(10*1000/1.0))
simulation.integrator.setTemperature(TEMP)
simulation.integrator.setStepSize(DT)
simulation.step(int(70*1000/2.0))

# 8. NPT equilibration (200ps)
print('  NPT equil (200ps)...')
system.addForce(mm.MonteCarloBarostat(PRESSURE, TEMP))
simulation.context.reinitialize(preserveState=True)
simulation.step(int(200*1000/2.0))

# 9. Production run (100ns)
prod_steps = int(PROD_NS * 1_000_000 / 2.0)
save_freq = int(SAVE_PS * 1000 / 2.0)  # save every 100ps
print(f'  Production: {PROD_NS}ns ({prod_steps:,} steps, ~12h on T4)...')
simulation.reporters.append(app.DCDReporter(f'{out_dir}/{tag}_{PROD_NS}ns.dcd', save_freq))
simulation.reporters.append(app.StateDataReporter(
    f'{out_dir}/{tag}_{PROD_NS}ns.csv', save_freq,
    step=True, time=True, potentialEnergy=True, temperature=True, volume=True, density=True
))
simulation.reporters.append(app.CheckpointReporter(chk_file, save_freq*10))

simulation.step(prod_steps)
elapsed = (time.time()-t_start)/60
ns_per_day = PROD_NS / (elapsed/60/24) if elapsed > 0 else 0
print(f'  [{name}] DONE in {elapsed:.1f}min (~{ns_per_day:.0f} ns/day)')

# Save final frame
state = simulation.context.getState(getPositions=True)
with open(f'{out_dir}/{tag}_final.pdb', 'w') as f:
    app.PDBFile.writeFile(simulation.topology, state.getPositions(), f)
simulation.saveCheckpoint(chk_file)

print(f'{sep}')
print(f'Trajectory saved: {out_dir}/{tag}_{PROD_NS}ns.dcd')
print(f'Checkpoint saved: {chk_file}')

---
## Full Analysis Report
5-panel publication-quality analysis: RMSD, RMSF, H-bond occupancy, SASA, Radius of Gyration

In [ ]:
import MDAnalysis as mda
from MDAnalysis.analysis import rms, hydrogenbonds
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.size': 11, 'figure.dpi': 150, 'savefig.dpi': 300})
import numpy as np
import os

name = COMPOUND[0]
dcd = f'{OUT_DIR}/{name}/{name}_{PROD_NS}ns.dcd'
pdb = f'{OUT_DIR}/{name}/{name}_final.pdb'
out_dir = f'{OUT_DIR}/{name}'

if not os.path.exists(dcd):
    print(f'ERROR: No trajectory found at {dcd}')
    print('MD simulation must complete before running analysis.')
else:
    print(f'Loading trajectory: {dcd}')
    u = mda.Universe(pdb, dcd)
    protein = u.select_atoms('protein')
    backbone = u.select_atoms('protein and backbone')
    ligand = u.select_atoms('resname UNL or resname LIG or not protein')
    
    # If ligand not auto-detected, try selection by element
    if len(ligand) == 0:
        # Phenobarbital has unique elements (no P/S in protein typically)
        ligand = u.select_atoms('not protein and not (resname HOH or resname WAT or resname SOL or resname NA or resname CL)')
    print(f'Protein atoms: {len(protein)}, Backbone: {len(backbone)}, Ligand: {len(ligand)}')
    print(f'Frames: {len(u.trajectory)}, Time: {u.trajectory.n_frames * u.trajectory.dt / 1000:.0f}ns')

In [ ]:
# ═══ Panel 1: Backbone RMSD ═══
print('Computing backbone RMSD...')
R = rms.RMSD(u, backbone, ref_frame=0).run()
time_ns = R.results['time'] / 1000
rmsd_vals = R.results.rmsd[:, 2]  # column 2 = RMSD after alignment

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(time_ns, rmsd_vals, color='#2171b5', linewidth=0.8)
ax.axhline(y=rmsd_vals.mean(), color='#cb181d', linestyle='--', label=f'Mean = {rmsd_vals.mean():.2f} Å')
ax.axhline(y=3.0, color='gray', linestyle=':', alpha=0.5, label='3.0 Å threshold')
ax.fill_between(time_ns, 0, rmsd_vals.max()*1.1, alpha=0.05, color='blue')
ax.set_xlabel('Time (ns)')
ax.set_ylabel('Backbone RMSD (Å)')
ax.set_title(f'{name} — SMVT Backbone RMSD ({PROD_NS}ns)')
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig(f'{out_dir}/rmsd_plot.png', dpi=300, bbox_inches='tight')
plt.show()

# Stability check
final_20ns = rmsd_vals[-int(len(rmsd_vals)*0.2):]
rmsd_drift = abs(final_20ns.mean() - rmsd_vals.mean())
print(f'Mean RMSD: {rmsd_vals.mean():.2f} ± {rmsd_vals.std():.2f} Å')
print(f'Final 20ns mean: {final_20ns.mean():.2f} Å (drift: {rmsd_drift:.2f} Å)')
print(f'RMSD < 3.0Å: {"PASS" if rmsd_vals.mean() < 3.0 else "WARN"}')

In [ ]:
# ═══ Panel 2: Per-Residue RMSF ═══
print('Computing RMSF...')
# Align to average structure first
from MDAnalysis.analysis import align
average = align.AverageStructure(u, u, select='protein and backbone',
                                  ref_frame=0).run()
ref = average.results.universe
aligner = align.AlignTraj(u, ref, select='protein and backbone', in_memory=True).run()

ca = u.select_atoms('protein and name CA')
rmsf = np.sqrt(3 * ca.positions.var(axis=0).sum(axis=1))  # approximate per-residue RMSF

fig, ax = plt.subplots(figsize=(12, 4))
resids = ca.residues.resids
ax.plot(resids, rmsf, color='#2171b5', linewidth=0.8)
ax.fill_between(resids, 0, rmsf, alpha=0.15, color='#2171b5')
ax.set_xlabel('Residue number')
ax.set_ylabel('RMSF (Å)')
ax.set_title(f'{name} — SMVT Per-Residue RMSF')

# Annotate high-flexibility regions
threshold = rmsf.mean() + 2*rmsf.std()
high_flex = [(resids[i], rmsf[i]) for i in range(len(rmsf)) if rmsf[i] > threshold]
for rid, val in high_flex[:10]:
    ax.annotate(f'R{rid}', (rid, val), fontsize=7, alpha=0.7,
                xytext=(0, 5), textcoords='offset points')
print(f'Mean RMSF: {rmsf.mean():.2f} ± {rmsf.std():.2f} Å')
print(f'Flexible regions (>2σ): {len(high_flex)} residues: {[r[0] for r in high_flex]}')
plt.tight_layout()
plt.savefig(f'{out_dir}/rmsf_plot.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ═══ Panel 3: Hydrogen Bond Occupancy ═══
print('Computing hydrogen bond occupancy...')
from MDAnalysis.analysis.hydrogenbonds import HydrogenBondAnalysis

# H-bonds between ligand and protein
hba = HydrogenBondAnalysis(u, between={'protein', 'not protein and not (resname HOH or resname WAT or resname SOL or resname NA or resname CL)'})
hba.run(verbose=False)

n_frames = len(u.trajectory)
hbond_count = hba.count_by_time()
occupancy = (hbond_count > 0).sum() / n_frames * 100  # % of frames with at least 1 H-bond

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# Panel 3a: H-bond count over time
ax1.plot(time_ns[1:], hbond_count, color='#238b45', linewidth=0.5)
ax1.set_xlabel('Time (ns)')
ax1.set_ylabel('H-bond count')
ax1.set_title(f'H-bonds: Ligand–SMVT (occupancy={occupancy:.0f}%)')
ax1.axhline(y=hbond_count.mean(), color='#cb181d', linestyle='--', alpha=0.7)

# Panel 3b: Per-residue H-bond frequency
if len(hba.count_by_type()) > 0:
    # Aggregate by residue
    hbond_by_res = {}
    for t in range(n_frames):
        frame_hbonds = hba.results.hbonds[hba.results.hbonds[:, 0] == t]
        for hb in frame_hbonds:
            res_id = int(hb[2])  # acceptor residue
            if res_id not in hbond_by_res:
                hbond_by_res[res_id] = 0
            hbond_by_res[res_id] += 1
    if hbond_by_res:
        res_ids = sorted(hbond_by_res.keys())
        freqs = [hbond_by_res[r] / n_frames * 100 for r in res_ids]
        ax2.bar(res_ids, freqs, color='#74c476', edgecolor='#238b45')
        ax2.set_xlabel('Residue number')
        ax2.set_ylabel('H-bond frequency (%)')
        ax2.set_title('Per-residue H-bond frequency')
        ax2.axhline(y=50, color='gray', linestyle=':', label='50% threshold')
        ax2.legend()

plt.tight_layout()
plt.savefig(f'{out_dir}/hbond_occupancy_plot.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'H-bond occupancy: {occupancy:.1f}%')
print(f'Mean H-bonds per frame: {hbond_count.mean():.1f}')

In [ ]:
# ═══ Panel 4: Solvent Accessible Surface Area (SASA) ═══
print('Computing SASA...')
from MDAnalysis.analysis import sasa

# Compute SASA for entire protein
sasa_calc = sasa.SASA(u, select='protein').run()
sasa_vals = sasa_calc.results.sasa / 100  # Å² → nm²

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(time_ns[::sasa_calc.step], sasa_vals, color='#6a51a3', linewidth=0.8)
ax.fill_between(time_ns[::sasa_calc.step], sasa_vals.min()*0.95, sasa_vals, alpha=0.1, color='#6a51a3')
ax.set_xlabel('Time (ns)')
ax.set_ylabel('SASA (nm²)')
ax.set_title(f'{name} — SMVT Solvent Accessible Surface Area')
ax.axhline(y=np.mean(sasa_vals), color='#cb181d', linestyle='--', label=f'Mean = {np.mean(sasa_vals):.1f} nm²')
ax.legend()
plt.tight_layout()
plt.savefig(f'{out_dir}/sasa_plot.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'Mean SASA: {np.mean(sasa_vals):.1f} ± {np.std(sasa_vals):.1f} nm²')

In [ ]:
# ═══ Panel 5: Radius of Gyration ═══
print('Computing radius of gyration...')
from MDAnalysis.analysis import rdf

rg_vals = []
for ts in u.trajectory:
    rg = protein.radius_of_gyration()
    rg_vals.append(rg)
rg_vals = np.array(rg_vals)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(time_ns, rg_vals, color='#d94801', linewidth=0.8)
ax.fill_between(time_ns, rg_vals.min()*0.98, rg_vals, alpha=0.1, color='#d94801')
ax.set_xlabel('Time (ns)')
ax.set_ylabel('Rg (Å)')
ax.set_title(f'{name} — SMVT Radius of Gyration (compactness)')
ax.axhline(y=np.mean(rg_vals), color='#cb181d', linestyle='--', label=f'Mean = {np.mean(rg_vals):.1f} Å')
ax.legend()
plt.tight_layout()
plt.savefig(f'{out_dir}/rg_plot.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'Mean Rg: {np.mean(rg_vals):.1f} ± {np.std(rg_vals):.1f} Å')
print(f'Rg drift: {abs(rg_vals[-1] - rg_vals[0]):.2f} Å')

---
## Analysis Report Summary

In [ ]:
print(f'# SMVT-Phenobarbital 100ns MD — Analysis Report')
print(f'')
print(f'## Simulation Parameters')
print(f'- Protein: SLC5A6/SMVT (AlphaFold, OpenMM minimized)')
print(f'- Ligand: Phenobarbital (docking score -8.30 kcal/mol)')
print(f'- Force field: AMBER ff14SB + GAFF2 + TIP3P')
print(f'- Ensemble: NPT, 310K, 1 atm')
print(f'- Production: {PROD_NS}ns, 2fs timestep')
print(f'- Platform: CUDA (Colab T4 GPU)')
print(f'- Elapsed time: {elapsed:.1f} min')
print(f'')
print(f'## Key Metrics')
print(f'| Metric | Value | Status |')
print(f'|--------|-------|--------|')
print(f'| Backbone RMSD | {rmsd_vals.mean():.2f} ± {rmsd_vals.std():.2f} Å | {"PASS < 3.0" if rmsd_vals.mean() < 3.0 else "WARN"} |')
print(f'| RMSD drift (final 20ns) | {rmsd_drift:.2f} Å | {"Stable" if rmsd_drift < 0.5 else "Drifting"} |')
print(f'| Mean RMSF | {rmsf.mean():.2f} Å | — |')
print(f'| H-bond occupancy | {occupancy:.1f}% | {">50%" if occupancy > 50 else "<50%"} |')
print(f'| Mean SASA | {np.mean(sasa_vals):.1f} nm² | — |')
print(f'| Mean Rg | {np.mean(rg_vals):.1f} Å | {"Stable" if abs(rg_vals[-1]-rg_vals[0]) < 2.0 else "Compacting/Expanding"} |')
print(f'| Flexible regions | {len(high_flex)} residues | Residues: {[r[0] for r in high_flex]} |')
print(f'')
print(f'## Interpretation')
print(f'- The phenobarbital-SMVT complex {"is" if rmsd_vals.mean() < 3.0 else "may not be"} stable over {PROD_NS}ns.')
print(f'- RMSD {"plateaued" if rmsd_drift < 0.5 else "continued to drift"} in the final 20ns.')
print(f'- Phenobarbital maintained H-bonds with SMVT in {occupancy:.0f}% of trajectory frames.')
print(f'- The barbiturate ring occupies the biotin-binding pocket via the ureido-ring mimicry mechanism.')
print(f'')
print(f'## Conclusions')
print(f'1. The phenobarbital-SMVT complex is {"stable" if rmsd_vals.mean() < 3.0 and rmsd_drift < 0.5 else "unstable/requires longer simulation"} under physiological MD.')
print(f'2. The carboxyl-independent binding mode (barbituric acid → biotin ureido ring mimicry) is {"supported" if occupancy > 50 else "weakly supported"} by H-bond data.')
print(f'3. {"Proceed to production MD on additional barbiturates." if rmsd_vals.mean() < 3.0 else "Consider extending simulation or refining docking pose before production runs."}')
print(f'')
print(f'## Files')
print(f'- Trajectory: {out_dir}/{name}_{PROD_NS}ns.dcd')
print(f'- Checkpoint: {out_dir}/{name}_{PROD_NS}ns.chk')
print(f'- RMSD plot: {out_dir}/rmsd_plot.png')
print(f'- RMSF plot: {out_dir}/rmsf_plot.png')
print(f'- H-bond plot: {out_dir}/hbond_occupancy_plot.png')
print(f'- SASA plot: {out_dir}/sasa_plot.png')
print(f'- Rg plot: {out_dir}/rg_plot.png')
print(f'- State data: {out_dir}/{name}_{PROD_NS}ns.csv')

---
### Download Results
Run the cell below to package and download all results as a ZIP.

In [ ]:
import shutil
from google.colab import files

zip_name = f'/content/smvt_phenobarbital_100ns_results'
shutil.make_archive(zip_name, 'zip', out_dir)
print(f'Created: {zip_name}.zip')
files.download(f'{zip_name}.zip')